# RBA 이상탐지 — self-contained 학습 (전처리·학습이 셀에 드러남)

github clone 없이 **각 셀에서 전처리·학습 과정을 직접** 보여준다. CPU IsolationForest라 GPU 불필요.

**규율(누수 차단):** 학습은 라벨을 쓰지 않는 unlabeled fit + 시간순 분할, **라벨은 평가에만**. 임계는 정상 held-out의 목표 FPR 분위수. 점수는 공격 확률이 아니고 evidence는 관측 대 기준 비교일 뿐이다.

데이터: Wiefling RBA 로그인 데이터셋(Zenodo 6782156, 실제 로그인). 오늘 밤 완주를 위해 `ROW_CAP`으로 앞부분만 스트리밍(전량은 3,126만 행).


## 0. 파라미터


In [ ]:
import sys, platform, time
print(platform.platform(), sys.version)

ROW_CAP     = 3_000_000    # 오늘 밤 완주용 상한(전량 학습은 None). 스트리밍이라 메모리 안전.
TARGET_FPR  = 0.01         # calibration 정상에서 이 비율만 임계 초과하도록
EMBARGO_S   = 3600         # 분할 경계 embargo(초)
SEED        = 42
ZENODO_ZIP  = "https://zenodo.org/records/6782156/files/rba-dataset.zip?download=1"  # md5 cc1b1078b3929650e6c08678caffcc57

## 1. 설치 (CPU 전용 — numpy, scikit-learn)


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","numpy","scikit-learn","joblib"], check=True)
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, roc_auc_score
print("ready")

## 2. 데이터 다운로드 (Zenodo RBA zip)


In [ ]:
import os, urllib.request, zipfile
ZIP = "/content/rba-dataset.zip"
if not os.path.exists(ZIP):
    print("downloading ~1GB ...", flush=True); urllib.request.urlretrieve(ZENODO_ZIP, ZIP)
z = zipfile.ZipFile(ZIP)
CSV_MEMBER = [n for n in z.namelist() if n.endswith(".csv")][0]
print("zip ok, csv member:", CSV_MEMBER, "size(MB):", round(os.path.getsize(ZIP)/1e6,1))

## 3. 전처리 — 로그인 이력 대비 11개 인과 피처 (누수 없음)

각 로그인의 피처는 **그 시점 이전 이력만으로** 계산하고 그 뒤 이력을 갱신한다(미래 누수 방지). 라벨: 1=ATO(양성), -1=공격IP지만 non-ATO(평가 제외), 0=정상.


In [ ]:
import csv, io, math
from collections import defaultdict, Counter
from datetime import datetime

NAMES = ["log_prior_attempts","first_observation","log_seconds_since_previous",
         "country_new","country_prior_frequency","ip_new","ip_prior_frequency",
         "device_new","device_prior_frequency","browser_new","browser_prior_frequency"]

def parse_ts(s):
    try: return datetime.fromisoformat(s.replace("Z","+00:00")).timestamp()
    except Exception:
        for f in ("%Y-%m-%d %H:%M:%S.%f","%Y-%m-%d %H:%M:%S"):
            try: return datetime.strptime(s, f).timestamp()
            except Exception: pass
    return None

# 사용자별 이력 상태(현재까지 관측). 피처는 갱신 '전'에 계산 → 인과성 보장.
state = defaultdict(lambda: {"n":0,"last":None,"co":Counter(),"ip":Counter(),"dv":Counter(),"br":Counter()})
X=[]; y=[]; t=[]
t0=time.time()
with z.open(CSV_MEMBER) as fh:
    reader = csv.DictReader(io.TextIOWrapper(fh, encoding="utf-8", newline=""))
    for i,row in enumerate(reader):
        if ROW_CAP and i>=ROW_CAP: break
        uid=row.get("User ID"); ts=parse_ts(row.get("Login Timestamp",""))
        if uid is None or ts is None: continue
        co=row.get("Country",""); ip=row.get("IP Address",""); dv=row.get("Device Type",""); br=row.get("Browser Name and Version","")
        st=state[uid]; n=st["n"]
        def nf(counter,val):
            seen=counter.get(val,0)
            return (1.0 if seen==0 else 0.0, (seen/n) if n>0 else 0.0)
        c_new,c_f=nf(st["co"],co); i_new,i_f=nf(st["ip"],ip); d_new,d_f=nf(st["dv"],dv); b_new,b_f=nf(st["br"],br)
        feats=[math.log1p(n), 1.0 if n==0 else 0.0,
               math.log1p(ts-st["last"]) if st["last"] is not None else 0.0,
               c_new,c_f,i_new,i_f,d_new,d_f,b_new,b_f]
        ato=str(row.get("Is Account Takeover","")).lower() in ("true","1")
        aip=str(row.get("Is Attack IP","")).lower() in ("true","1")
        lab = 1 if ato else (-1 if aip else 0)
        X.append(feats); y.append(lab); t.append(ts)
        # 이력 갱신(피처 계산 후)
        st["n"]=n+1; st["last"]=ts; st["co"][co]+=1; st["ip"][ip]+=1; st["dv"][dv]+=1; st["br"][br]+=1
        if i and i%500000==0: print(f"  {i} rows, {time.time()-t0:.0f}s", flush=True)

X=np.asarray(X,dtype=np.float32); y=np.asarray(y,dtype=np.int8); t=np.asarray(t,dtype=np.float64)
print("rows:",len(X)," users:",len(state)," label counts -> 0:",int((y==0).sum())," 1(ATO):",int((y==1).sum())," -1:",int((y==-1).sum())," prep_s:",round(time.time()-t0))

## 4. 시간순 분할 + embargo (train < cal < test)

과거로 배우고 미래를 평가. 경계마다 embargo로 인접 구간을 버려 train/test가 시간적으로 겹치지 않게 한다. **라벨로 행을 지우지 않는다**(unlabeled).


In [ ]:
order=np.argsort(t, kind="stable")
X,y,t = X[order],y[order],t[order]
tmin,tmax=t[0],t[-1]
b1=tmin+(tmax-tmin)*0.60      # train | cal
b2=tmin+(tmax-tmin)*0.80      # cal   | test
train = t <  b1
cal   = (t>=b1+EMBARGO_S) & (t<b2)
test  =  t>=b2+EMBARGO_S
for nm,m in [("train",train),("cal",cal),("test",test)]:
    print(f"{nm}: {int(m.sum())} rows | ATO(1): {int((y[m]==1).sum())} | normal(0): {int((y[m]==0).sum())}")

## 5. IsolationForest 학습 — train에 unlabeled fit

라벨을 쓰지 않고 train 분포만으로 '정상 다양체'를 학습. 점수 = `-decision_function`(클수록 이상).


In [ ]:
clf = IsolationForest(n_estimators=200, max_samples=512, contamination="auto",
                       random_state=SEED, n_jobs=-1)
clf.fit(X[train])                      # unlabeled: 라벨 미사용
score = -clf.decision_function(X)      # 방향 고정: 클수록 이상
print("fitted on", int(train.sum()), "train rows (unlabeled)")

## 6. 임계(calibration 정상) + 평가(라벨은 여기서만)

임계는 calibration의 **정상(0)** 점수 분포에서 목표 FPR 분위수로. 그다음 test에서 recall/FPR/PR-AUC/ROC-AUC + Wilson95 CI. 점수↔공격확률 아님, evidence↔인과 아님.


In [ ]:
def wilson(k,n,z=1.96):
    if n==0: return (0.0,0.0)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n); m=z*math.sqrt((p*(1-p)+z*z/(4*n))/n)
    return ((c-m)/d,(c+m)/d)

cal_norm = score[cal & (y==0)]
thr = float(np.quantile(cal_norm, 1-TARGET_FPR)) if len(cal_norm) else 0.0
flag = score > thr

def report(mask, name):
    s,lab = score[mask], y[mask]
    pos, neg = (lab==1), (lab==0)
    tp=int((flag[mask]&pos).sum()); fn=int((~flag[mask]&pos).sum())
    fp=int((flag[mask]&neg).sum()); tn=int((~flag[mask]&neg).sum())
    P=int(pos.sum()); N=int(neg.sum())
    rec = tp/P if P else None; fpr = fp/N if N else None
    lo,hi = wilson(tp,P) if P else (None,None)
    # PR/ROC는 라벨 {0,1}만(–1 제외), 양쪽 클래스 있을 때만
    keep = pos|neg; prauc=rocauc=None
    if P>0 and N>0:
        prauc=round(float(average_precision_score(lab[keep]==1, s[keep])),4)
        rocauc=round(float(roc_auc_score(lab[keep]==1, s[keep])),4)
    print(f"[{name}] P(ATO)={P} N(normal)={N} | recall={rec} (Wilson95 {None if lo is None else round(lo,3)}~{None if hi is None else round(hi,3)}) | FPR={None if fpr is None else round(fpr,4)} | PR-AUC={prauc} ROC-AUC={rocauc} | flagged={int(flag[mask].sum())}")

print("threshold(GT):", round(thr,6), " target_fpr:", TARGET_FPR)
report(test, "test(시간분할·정직)")
report((y!=-1), "pooled(설계검증·전체 라벨)")
print("주의: test 양성이 0이면 recall 미성립 → pooled는 설계검증용, 시간분할 test가 일반화 지표.")

## 7. 모델 저장 + 다음 단계


In [ ]:
import joblib, json
bundle={"model":clf,"threshold":thr,"threshold_operator":"GT","feature_names":NAMES,
        "feature_version":"rba-login-causal-11-v1","target_fpr":TARGET_FPR,"row_cap":ROW_CAP,"seed":SEED}
joblib.dump(bundle, "/content/rba_iforest.joblib")
print("saved /content/rba_iforest.joblib")
from google.colab import files
files.download("/content/rba_iforest.joblib")
print("다음: 이 임계·성능을 ZETTY에 그대로 이식 X. ZETTY 자체 정상으로 임계 재보정 + s4_ground_truth로 재측정(공개셋 수치와 분리 보고).")

## 정직 노트

- 점수는 공격 확률이 아니다. 요청량 증가를 관측했다고 그것이 점수의 원인이라 단정하지 않는다(관측 ≠ 인과).
- 라벨은 평가에만 썼고, 학습은 unlabeled + 시간분할이다(누수 차단).
- 극소 양성이라 recall은 Wilson95 CI와 함께 본다. ROW_CAP 앞부분만 쓰면 test 양성이 0일 수 있다 → pooled는 설계검증, 시간분할 test가 일반화 지표.
- 공개셋 성능 ≠ ZETTY 성능. 임계는 ZETTY 정상으로 재보정, 성능은 s4_ground_truth로 재측정.
